# ShiftLog-Gym GRPO Training (Colab)

This notebook keeps training separate from the environment repo and uses the TRL `environment_factory` flow.

In [ ]:
# Recommended in Colab:
# !pip install -q trl transformers datasets peft accelerate bitsandbytes matplotlib pandas


In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
FALLBACK_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'


In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

from shiftlog_gym.trl_env import ShiftLogToolEnv, reward_recall, reward_total


In [ ]:
prompt = (
    'You are the primary SRE on call. Use the shift log intentionally. '
    'Before acting on incidents that may be repeats or causal follow-ons, retrieve relevant memory. '
    'Write compact, structured facts to the log. Avoid contradictions and unsupported mitigations.'
)

families = ['db_pool_exhaustion', 'auth_timeout_cascade', 'memory_oom_signature', 'feature_flag_regression']
dataset = Dataset.from_dict({
    'prompt': [[{'role': 'user', 'content': prompt}]] * len(families),
    'family': families,
    'variant_index': [0, 0, 0, 0],
})
dataset


In [ ]:
args = GRPOConfig(
    output_dir='shiftlog-grpo',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    max_completion_length=2048,
    logging_steps=1,
    save_steps=20,
    report_to=[],
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    train_dataset=dataset,
    args=args,
    reward_funcs=[reward_total, reward_recall],
    environment_factory=ShiftLogToolEnv,
)

# trainer.train()


## What To Track

Export and show at minimum:

- total reward curve
- `R_recall` curve
- early `read_shift_log` frequency on linked incidents
- linked incident success rate
- contradiction / bad-write rate
